In [46]:
#Import Packages
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [47]:
# Load data
file_path = "meal_deal_data.xlsx"
df = pd.read_excel(file_path, sheet_name="Sheet1").copy()

# Clean column names
df.columns = (
    df.columns
    .str.strip()
    .str.replace(" ", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace("/", "_", regex=False)
)

# Rename a few columns to make coding easier
df = df.rename(columns={
    "Individual_Price": "price",
    "Individual Price": "price",
    "EnergyCalories": "calories",
    "Energy(Calories)": "calories",
    "Fat_g": "fat",
    "Fat (g)": "fat",
    "Saturates_Fat_g": "sat_fat",
    "Saturates Fat (g)": "sat_fat",
    "Carbohydrates_g": "carbs",
    "Carbohydrates (g) ": "carbs",
    "Sugars_g": "sugar",
    "Sugars (g)": "sugar",
    "Fibre_g": "fibre",
    "Fibre (g)": "fibre",
    "Protein_g": "protein",
    "Protein (g)": "protein",
    "Salt_g": "salt",
    "Salt (g)": "salt",
    "Prime5.5GBP": "is_prime",
    "Prime(5.5GBP)": "is_prime",
    "Category": "category",
    "Item": "item",
    "Vegetarian": "vegetarian",
    "Vegan": "vegan",
    "Dairy": "dairy",
    "Gluten": "gluten",
    "Porks": "pork",
    "Beef": "beef",
    "Chicken": "chicken",
    "Fish": "fish",
    "Sweet": "sweet",
    "Salty": "salty",
    "Wrap": "wrap",
    "Sandwich": "sandwich",
    "Sushi": "sushi",
    "Bowl": "bowl",
    "Western": "western",
    "Asian": "asian",
    "Mediterranean": "mediterranean",
    "Indian": "indian",
    "Caribbean": "caribbean"
})

# Standardize category labels
df["category"] = df["category"].astype(str).str.strip().str.title()

# Convert numeric columns
numeric_cols = [
    "price", "calories", "fat", "sat_fat", "carbs", "sugar",
    "fibre", "protein", "salt"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# Convert binary columns
binary_cols = [
    "is_prime", "vegetarian", "vegan", "dairy", "gluten",
    "pork", "beef", "chicken", "fish",
    "sweet", "salty",
    "wrap", "sandwich", "sushi", "bowl",
    "western", "asian", "mediterranean", "indian", "caribbean"
]

for col in binary_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# Split into the three meal-deal categories
mains = df[df["category"] == "Main"].reset_index(drop=True)
snacks = df[df["category"] == "Snack"].reset_index(drop=True)
drinks = df[df["category"] == "Drink"].reset_index(drop=True)

print("Number of mains:", len(mains))
print("Number of snacks:", len(snacks))
print("Number of drinks:", len(drinks))

display(df.head())

Number of mains: 94
Number of snacks: 46
Number of drinks: 31


,item,category,price,calories,fat,sat_fat,carbs,sugar,fibre,protein,...,salty,wrap,sandwich,sushi,bowl,western,asian,mediterranean,indian,caribbean
0,Tony's Chocolonely Milk Caramel Seasalt 47g,Snack,1.35,252.0,15.0,9.0,25.0,24.0,0.0,3.3,...,1,0,0,0,0,0,0,0,0,0
1,Pepsi regular soft drink 500ml,Drink,1.95,92.0,0.0,0.0,22.0,22.0,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
2,Tesco Fully Loaded Korean Style Pulled Pork Pr...,Main,4.50,669.0,29.3,11.3,69.8,19.3,3.0,30.3,...,0,0,0,0,1,1,0,0,0,0
3,Trek Protein Flapjack with Lotus Biscoff 50g,Snack,1.85,230.0,11.0,4.4,23.0,12.0,2.2,9.8,...,0,0,0,0,0,0,0,0,0,0
4,Tesco Chicken Tikka & Mango Chutney Sandwich,Main,2.75,333.0,4.6,1.0,49.0,8.6,4.2,21.9,...,0,0,1,0,0,0,0,0,1,0


In [48]:
#Define user-adjustable inputs

params = {
    # choose ONE objective each time
    "objective": "max_protein",  
    # options later can be:
    # "max_protein", "min_sugar", "min_fat", "min_carbs",
    # "min_calories", "min_salt", "max_savings"

    # Tesco meal deal prices
    "regular_price": 3.85,
    "prime_price": 5.50,

    # dietary constraints
    "vegetarian_only": False,
    "vegan_only": False,
    "no_pork": False,
    "no_beef": False,
    "no_chicken": False,
    "no_fish": False,
    "gluten_free": False,
    "dairy_free": False,

    # NEW preference parameters
    # protein choice: "any", "pork", "beef", "chicken", "fish", "others"
    "protein_preference": "any",

    # snack type: "no_preference", "sweet", "salty", "both"
    "snack_preference": "no_preference",

    # main type: "no_preference", "wrap", "sandwich", "sushi", "bowl"
    "main_type_preference": "no_preference",

    # cuisine: "no_preference", "western", "asian", "mediterranean", "indian", "caribbean"
    "cuisine_preference": "no_preference",

    # options: "no_preference", "prime_only", "non_prime_only"
    "prime_preference": "no_preference" ,
    

    # nutrition constraints
    "min_protein": None,
    "max_protein": None,
    "min_calories": None,
    "max_calories": None,
    "min_sugar": None,
    "max_sugar": None,
    "min_fat": None,
    "max_fat": None,
    "min_carbs": None,
    "max_carbs": None,
    "min_salt": None,
    "max_salt": None,

    # category-specific nutrition constraints
    "max_snack_calories": None,
    "max_drink_calories": None
}

In [49]:
#Start building the model

def solve_meal_deal(mains, snacks, drinks, params):
    model = gp.Model("tesco_meal_deal")

    # Index sets
    M = mains.index.tolist()
    S = snacks.index.tolist()
    D = drinks.index.tolist()

    mains = mains.copy()
    snacks = snacks.copy()
    drinks = drinks.copy()  

    mains["other_protein"] = (
        mains[["pork", "beef", "chicken", "fish"]].sum(axis=1) == 0
    ).astype(int)

    snacks["sweet_and_salty"] = (
        (snacks["sweet"] == 1) & (snacks["salty"] == 1)
    ).astype(int)
    # Decision variables
    x = model.addVars(M, vtype=GRB.BINARY, name="main")
    y = model.addVars(S, vtype=GRB.BINARY, name="snack")
    z = model.addVars(D, vtype=GRB.BINARY, name="drink")

    # Exactly one main, one snack, one drink
    model.addConstr(gp.quicksum(x[i] for i in M) == 1, name="one_main")
    model.addConstr(gp.quicksum(y[j] for j in S) == 1, name="one_snack")
    model.addConstr(gp.quicksum(z[k] for k in D) == 1, name="one_drink")

    # Total nutrition expressions
    total_calories = (
        gp.quicksum(mains.loc[i, "calories"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "calories"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "calories"] * z[k] for k in D)
    )

    total_protein = (
        gp.quicksum(mains.loc[i, "protein"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "protein"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "protein"] * z[k] for k in D)
    )

    total_sugar = (
        gp.quicksum(mains.loc[i, "sugar"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "sugar"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "sugar"] * z[k] for k in D)
    )

    total_fat = (
        gp.quicksum(mains.loc[i, "fat"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "fat"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "fat"] * z[k] for k in D)
    )

    total_carbs = (
        gp.quicksum(mains.loc[i, "carbs"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "carbs"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "carbs"] * z[k] for k in D)
    )

    total_salt = (
        gp.quicksum(mains.loc[i, "salt"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "salt"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "salt"] * z[k] for k in D)
    )

    total_individual_price = (
        gp.quicksum(mains.loc[i, "price"] * x[i] for i in M) +
        gp.quicksum(snacks.loc[j, "price"] * y[j] for j in S) +
        gp.quicksum(drinks.loc[k, "price"] * z[k] for k in D)
    )

    # Prime status depends ONLY on selected main
    selected_main_is_prime = gp.quicksum(mains.loc[i, "is_prime"] * x[i] for i in M)

    # Bundle price rule:
    # if selected main is prime -> 5.5
    # else -> 3.85
    meal_deal_price = (
        params["regular_price"] +
        (params["prime_price"] - params["regular_price"]) * selected_main_is_prime
    )

    total_savings = total_individual_price - meal_deal_price

    # -------------------------
    # Dietary constraints
    # -------------------------
    if params["vegetarian_only"]:
        model.addConstr(
            gp.quicksum((1 - mains.loc[i, "vegetarian"]) * x[i] for i in M) == 0,
            name="vegetarian_main"
        )
        model.addConstr(
            gp.quicksum((1 - snacks.loc[j, "vegetarian"]) * y[j] for j in S) == 0,
            name="vegetarian_snack"
        )
        model.addConstr(
            gp.quicksum((1 - drinks.loc[k, "vegetarian"]) * z[k] for k in D) == 0,
            name="vegetarian_drink"
        )

    if params["vegan_only"]:
        model.addConstr(
            gp.quicksum((1 - mains.loc[i, "vegan"]) * x[i] for i in M) == 0,
            name="vegan_main"
        )
        model.addConstr(
            gp.quicksum((1 - snacks.loc[j, "vegan"]) * y[j] for j in S) == 0,
            name="vegan_snack"
        )
        model.addConstr(
            gp.quicksum((1 - drinks.loc[k, "vegan"]) * z[k] for k in D) == 0,
            name="vegan_drink"
        )

    if params["no_pork"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "pork"] * x[i] for i in M) == 0,
            name="no_pork_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "pork"] * y[j] for j in S) == 0,
            name="no_pork_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "pork"] * z[k] for k in D) == 0,
            name="no_pork_drink"
        )

    if params["no_beef"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "beef"] * x[i] for i in M) == 0,
            name="no_beef_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "beef"] * y[j] for j in S) == 0,
            name="no_beef_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "beef"] * z[k] for k in D) == 0,
            name="no_beef_drink"
        )

    if params["no_chicken"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "chicken"] * x[i] for i in M) == 0,
            name="no_chicken_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "chicken"] * y[j] for j in S) == 0,
            name="no_chicken_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "chicken"] * z[k] for k in D) == 0,
            name="no_chicken_drink"
        )

    if params["no_fish"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "fish"] * x[i] for i in M) == 0,
            name="no_fish_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "fish"] * y[j] for j in S) == 0,
            name="no_fish_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "fish"] * z[k] for k in D) == 0,
            name="no_fish_drink"
        )

    if params["gluten_free"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "gluten"] * x[i] for i in M) == 0,
            name="gluten_free_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "gluten"] * y[j] for j in S) == 0,
            name="gluten_free_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "gluten"] * z[k] for k in D) == 0,
            name="gluten_free_drink"
        )

    if params["dairy_free"]:
        model.addConstr(
            gp.quicksum(mains.loc[i, "dairy"] * x[i] for i in M) == 0,
            name="dairy_free_main"
        )
        model.addConstr(
            gp.quicksum(snacks.loc[j, "dairy"] * y[j] for j in S) == 0,
            name="dairy_free_snack"
        )
        model.addConstr(
            gp.quicksum(drinks.loc[k, "dairy"] * z[k] for k in D) == 0,
            name="dairy_free_drink"
        )

    # -------------------------
    # Protein preference on MAIN
    # -------------------------
    protein_pref = params.get("protein_preference", "any")

    if protein_pref == "pork":
        model.addConstr(
            gp.quicksum(mains.loc[i, "pork"] * x[i] for i in M) == 1,
            name="prefer_pork_main"
        )

    elif protein_pref == "beef":
        model.addConstr(
            gp.quicksum(mains.loc[i, "beef"] * x[i] for i in M) == 1,
            name="prefer_beef_main"
        )

    elif protein_pref == "chicken":
        model.addConstr(
            gp.quicksum(mains.loc[i, "chicken"] * x[i] for i in M) == 1,
            name="prefer_chicken_main"
        )

    elif protein_pref == "fish":
        model.addConstr(
            gp.quicksum(mains.loc[i, "fish"] * x[i] for i in M) == 1,
            name="prefer_fish_main"
        )

    elif protein_pref == "others":
        model.addConstr(
            gp.quicksum(mains.loc[i, "other_protein"] * x[i] for i in M) == 1,
            name="prefer_other_protein_main"
        )

    # -------------------------
    # Snack type preference
    # -------------------------
    snack_pref = params.get("snack_preference", "no_preference")

    if snack_pref == "sweet":
        model.addConstr(
            gp.quicksum(snacks.loc[j, "sweet"] * y[j] for j in S) == 1,
            name="sweet_snack"
        )

    elif snack_pref == "salty":
        model.addConstr(
            gp.quicksum(snacks.loc[j, "salty"] * y[j] for j in S) == 1,
            name="salty_snack"
        )

    elif snack_pref == "both":
        model.addConstr(
            gp.quicksum(snacks.loc[j, "sweet_and_salty"] * y[j] for j in S) == 1,
            name="both_sweet_salty_snack"
        )

    # -------------------------
    # Main type preference
    # -------------------------
    main_type_pref = params.get("main_type_preference", "no_preference")

    if main_type_pref == "wrap":
        model.addConstr(
            gp.quicksum(mains.loc[i, "wrap"] * x[i] for i in M) == 1,
            name="main_wrap"
        )

    elif main_type_pref == "sandwich":
        model.addConstr(
            gp.quicksum(mains.loc[i, "sandwich"] * x[i] for i in M) == 1,
            name="main_sandwich"
        )

    elif main_type_pref == "sushi":
        model.addConstr(
            gp.quicksum(mains.loc[i, "sushi"] * x[i] for i in M) == 1,
            name="main_sushi"
        )

    elif main_type_pref == "bowl":
        model.addConstr(
            gp.quicksum(mains.loc[i, "bowl"] * x[i] for i in M) == 1,
            name="main_bowl"
        )

    # -------------------------
    # Main cuisine preference
    # -------------------------
    cuisine_pref = params.get("cuisine_preference", "no_preference")

    if cuisine_pref == "western":
        model.addConstr(
            gp.quicksum(mains.loc[i, "western"] * x[i] for i in M) == 1,
            name="main_western"
        )

    elif cuisine_pref == "asian":
        model.addConstr(
            gp.quicksum(mains.loc[i, "asian"] * x[i] for i in M) == 1,
            name="main_asian"
        )

    elif cuisine_pref == "mediterranean":
        model.addConstr(
            gp.quicksum(mains.loc[i, "mediterranean"] * x[i] for i in M) == 1,
            name="main_mediterranean"
        )

    elif cuisine_pref == "indian":
        model.addConstr(
            gp.quicksum(mains.loc[i, "indian"] * x[i] for i in M) == 1,
            name="main_indian"
        )

    elif cuisine_pref == "caribbean":
        model.addConstr(
            gp.quicksum(mains.loc[i, "caribbean"] * x[i] for i in M) == 1,
            name="main_caribbean"
        )

    # -------------------------
    # Prime constraints
    # -------------------------
    prime_pref = params.get("prime_preference", "no_preference")

    if prime_pref == "prime_only":
        model.addConstr(
            gp.quicksum(mains.loc[i, "is_prime"] * x[i] for i in M) == 1,
            name="prime_main_only"
        )

    elif prime_pref == "non_prime_only":
            model.addConstr(
            gp.quicksum((1 - mains.loc[i, "is_prime"]) * x[i] for i in M) == 1,
            name="non_prime_main_only"
        )
    # -------------------------
    # Nutrition constraints
    # -------------------------
    if params["min_protein"] is not None:
        model.addConstr(total_protein >= params["min_protein"], name="min_protein")

    if params["max_protein"] is not None:
        model.addConstr(total_protein <= params["max_protein"], name="max_protein")

    if params["min_calories"] is not None:
        model.addConstr(total_calories >= params["min_calories"], name="min_calories")

    if params["max_calories"] is not None:
        model.addConstr(total_calories <= params["max_calories"], name="max_calories")

    if params["min_sugar"] is not None:
        model.addConstr(total_sugar >= params["min_sugar"], name="min_sugar")

    if params["max_sugar"] is not None:
        model.addConstr(total_sugar <= params["max_sugar"], name="max_sugar")

    if params["min_fat"] is not None:
        model.addConstr(total_fat >= params["min_fat"], name="min_fat")

    if params["max_fat"] is not None:
        model.addConstr(total_fat <= params["max_fat"], name="max_fat")

    if params["min_carbs"] is not None:
        model.addConstr(total_carbs >= params["min_carbs"], name="min_carbs")

    if params["max_carbs"] is not None:
        model.addConstr(total_carbs <= params["max_carbs"], name="max_carbs")

    if params["min_salt"] is not None:
        model.addConstr(total_salt >= params["min_salt"], name="min_salt")

    if params["max_salt"] is not None:
        model.addConstr(total_salt <= params["max_salt"], name="max_salt")

    if params["max_snack_calories"] is not None:
        snack_calories = gp.quicksum(snacks.loc[j, "calories"] * y[j] for j in S)
        model.addConstr(snack_calories <= params["max_snack_calories"], name="max_snack_calories")

    if params["max_drink_calories"] is not None:
        drink_calories = gp.quicksum(drinks.loc[k, "calories"] * z[k] for k in D)
        model.addConstr(drink_calories <= params["max_drink_calories"], name="max_drink_calories")

    # -------------------------
    # Objective
    # -------------------------
    obj = params["objective"]

    if obj == "max_protein":
        model.setObjective(total_protein, GRB.MAXIMIZE)

    elif obj == "min_sugar":
        model.setObjective(total_sugar, GRB.MINIMIZE)

    elif obj == "min_fat":
        model.setObjective(total_fat, GRB.MINIMIZE)

    elif obj == "min_carbs":
        model.setObjective(total_carbs, GRB.MINIMIZE)

    elif obj == "min_calories":
        model.setObjective(total_calories, GRB.MINIMIZE)

    elif obj == "min_salt":
        model.setObjective(total_salt, GRB.MINIMIZE)

    elif obj == "max_savings":
        model.setObjective(total_savings, GRB.MAXIMIZE)

    else:
        raise ValueError(f"Unknown objective: {obj}")
    
        # Solve
    model.optimize()

    if model.Status != GRB.OPTIMAL:
        print("No optimal solution found.")
        print("Model status:", model.Status)
        return None

    # Extract selected items
    selected_main_idx = [i for i in M if x[i].X > 0.5][0]
    selected_snack_idx = [j for j in S if y[j].X > 0.5][0]
    selected_drink_idx = [k for k in D if z[k].X > 0.5][0]

    selected_main = mains.loc[selected_main_idx]
    selected_snack = snacks.loc[selected_snack_idx]
    selected_drink = drinks.loc[selected_drink_idx]

    result = {
        "main": selected_main["item"],
        "snack": selected_snack["item"],
        "drink": selected_drink["item"],
        "main_price": float(selected_main["price"]),
        "snack_price": float(selected_snack["price"]),
        "drink_price": float(selected_drink["price"]),
        "main_is_prime": int(selected_main["is_prime"]),
        "meal_deal_price": meal_deal_price.getValue(),
        "total_individual_price": total_individual_price.getValue(),
        "total_savings": total_savings.getValue(),
        "total_calories": total_calories.getValue(),
        "total_protein": total_protein.getValue(),
        "total_sugar": total_sugar.getValue(),
        "total_fat": total_fat.getValue(),
        "total_carbs": total_carbs.getValue(),
        "total_salt": total_salt.getValue(),
        "objective_used": obj,
        "objective_value": model.ObjVal,
    }

    return result

In [50]:
result = solve_meal_deal(mains, snacks, drinks, params)
result

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 171 columns and 171 nonzeros (Max)
Model fingerprint: 0xb0b06105
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 33.6000000
Presolve removed 3 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 76 33.6 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.600000000000e+01, best bound 7.600000000000e+01, gap 0.0000%


{'main': 'Zinda Chicken Makhni Air Wrap 225g',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.5,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 1,
 'meal_deal_price': 5.5,
 'total_individual_price': 7.15,
 'total_savings': 1.65,
 'total_calories': 731.0,
 'total_protein': 76.0,
 'total_sugar': 24.8,
 'total_fat': 15.9,
 'total_carbs': 71.4,
 'total_salt': 1.8,
 'objective_used': 'max_protein',
 'objective_value': 76.0}

In [51]:
#Test1
params.update({
    "objective": "max_protein",

    "protein_preference": "any",
    "snack_preference": "no_preference",
    "main_type_preference": "no_preference",
    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference",

    "max_calories": None
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 3 rows, 171 columns and 171 nonzeros (Max)
Model fingerprint: 0xb0b06105
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 33.6000000
Presolve removed 3 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 76 33.6 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.600000000000e+01, best bound 7.600000000000e+01, gap 0.0000%


{'main': 'Zinda Chicken Makhni Air Wrap 225g',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.5,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 1,
 'meal_deal_price': 5.5,
 'total_individual_price': 7.15,
 'total_savings': 1.65,
 'total_calories': 731.0,
 'total_protein': 76.0,
 'total_sugar': 24.8,
 'total_fat': 15.9,
 'total_carbs': 71.4,
 'total_salt': 1.8,
 'objective_used': 'max_protein',
 'objective_value': 76.0}

In [52]:
params.update({
    "objective": "max_protein",

    "max_calories": 700,

    "protein_preference": "any",
    "snack_preference": "no_preference",
    "main_type_preference": "no_preference",
    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 171 columns and 342 nonzeros (Max)
Model fingerprint: 0x1c0b0a9c
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Found heuristic solution: objective 15.7000000
Presolve removed 4 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 75 15.7 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+01, best bound 7.500000000000e+01, gap 0.0000%


{'main': 'The Gym Kitchen Chicken Tikka 400g',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.0,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 6.65,
 'total_savings': 2.8,
 'total_calories': 658.0,
 'total_protein': 75.0,
 'total_sugar': 26.7,
 'total_fat': 7.7,
 'total_carbs': 64.4,
 'total_salt': 1.67,
 'objective_used': 'max_protein',
 'objective_value': 75.0}

In [53]:
params.update({
    "protein_preference": "chicken",

    "snack_preference": "no_preference",
    "main_type_preference": "no_preference",
    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 390 nonzeros (Max)
Model fingerprint: 0xb404e7bc
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Found heuristic solution: objective 15.7000000
Presolve removed 5 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 75 15.7 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+01, best bound 7.500000000000e+01, gap 0.0000%


{'main': 'The Gym Kitchen Chicken Tikka 400g',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.0,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 6.65,
 'total_savings': 2.8,
 'total_calories': 658.0,
 'total_protein': 75.0,
 'total_sugar': 26.7,
 'total_fat': 7.7,
 'total_carbs': 64.4,
 'total_salt': 1.67,
 'objective_used': 'max_protein',
 'objective_value': 75.0}

In [54]:
params.update({
    "protein_preference": "any",

    "snack_preference": "sweet",

    "main_type_preference": "no_preference",
    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 358 nonzeros (Max)
Model fingerprint: 0x3c4b9027
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Found heuristic solution: objective 10.4000000
Presolve removed 5 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 75 10.4 

Optimal solution found (tolerance 1.00e-04)
Best objective 7.500000000000e+01, best bound 7.500000000000e+01, gap 0.0000%


{'main': 'The Gym Kitchen Chicken Tikka 400g',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.0,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 6.65,
 'total_savings': 2.8,
 'total_calories': 658.0,
 'total_protein': 75.0,
 'total_sugar': 26.7,
 'total_fat': 7.7,
 'total_carbs': 64.4,
 'total_salt': 1.67,
 'objective_used': 'max_protein',
 'objective_value': 75.0}

In [55]:
params.update({
    "protein_preference": "any",

    "snack_preference": "no_preference",

    "main_type_preference": "wrap",

    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 365 nonzeros (Max)
Model fingerprint: 0x961ee773
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Presolve removed 5 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 69.5 

Optimal solution found (tolerance 1.00e-04)
Best objective 6.950000000000e+01, best bound 6.950000000000e+01, gap 0.0000%


{'main': 'Zinda Chicken Makhni Air Wrap 225g',
 'snack': 'Fridge Raiders Grills Roast Chicken Fillet 35g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.5,
 'snack_price': 1.35,
 'drink_price': 2.25,
 'main_is_prime': 1,
 'meal_deal_price': 5.5,
 'total_individual_price': 7.1,
 'total_savings': 1.6,
 'total_calories': 663.0,
 'total_protein': 69.5,
 'total_sugar': 18.6,
 'total_fat': 15.8,
 'total_carbs': 62.0,
 'total_salt': 1.9600000000000002,
 'objective_used': 'max_protein',
 'objective_value': 69.5}

In [56]:
params.update({
    "protein_preference": "any",

    "snack_preference": "no_preference",

    "main_type_preference": "no_preference",

    "cuisine_preference": "asian",

    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 365 nonzeros (Max)
Model fingerprint: 0xd0ae8e57
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Found heuristic solution: objective 15.7000000
Presolve removed 2 rows and 161 columns
Presolve time: 0.00s
Presolved: 3 rows, 10 columns, 18 nonzeros
Found heuristic solution: objective 29.7000000
Variable types: 0 continuous, 10 integer (10 binary)
Found heuristic solution: objective 33.2000000

Root relaxation: objective 6.300000e+01, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Exp

{'main': 'The Gym Kitchen Katsu Chicken Wrap',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.0,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 6.65,
 'total_savings': 2.8,
 'total_calories': 639.0,
 'total_protein': 63.0,
 'total_sugar': 27.6,
 'total_fat': 9.200000000000001,
 'total_carbs': 73.4,
 'total_salt': 1.37,
 'objective_used': 'max_protein',
 'objective_value': 63.0}

In [57]:
params.update({
    "protein_preference": "any",

    "snack_preference": "no_preference",

    "main_type_preference": "no_preference",

    "cuisine_preference": "no_preference",

    "prime_preference": "prime_only"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 358 nonzeros (Max)
Model fingerprint: 0x91a1610d
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Found heuristic solution: objective 15.7000000
Presolve removed 5 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 69.5 15.7 

Optimal solution found (tolerance 1.00e-04)
Best objective 6.950000000000e+01, best bound 6.950000000000e+01, gap 0.0000%


{'main': 'Zinda Chicken Makhni Air Wrap 225g',
 'snack': 'Fridge Raiders Grills Roast Chicken Fillet 35g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 3.5,
 'snack_price': 1.35,
 'drink_price': 2.25,
 'main_is_prime': 1,
 'meal_deal_price': 5.5,
 'total_individual_price': 7.1,
 'total_savings': 1.6,
 'total_calories': 663.0,
 'total_protein': 69.5,
 'total_sugar': 18.6,
 'total_fat': 15.8,
 'total_carbs': 62.0,
 'total_salt': 1.9600000000000002,
 'objective_used': 'max_protein',
 'objective_value': 69.5}

In [58]:
params.update({
    "objective": "max_protein",

    "protein_preference": "fish",

    "snack_preference": "salty",

    "main_type_preference": "sushi",

    "cuisine_preference": "asian",

    "prime_preference": "no_preference",

    "max_calories": 750
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 8 rows, 171 columns and 422 nonzeros (Max)
Model fingerprint: 0x3f303092
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]

Found heuristic solution: objective 17.7000000
Presolve removed 8 rows and 171 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 51.4 17.7 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.140000000000e+01, best bound 5.140000000000e+01, gap 0.0000%


{'main': 'Tesco Finest Salmon Sushi Selection 196g',
 'snack': 'Tesco Egg & Salad Cream 130g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 5.0,
 'snack_price': 1.3,
 'drink_price': 2.25,
 'main_is_prime': 1,
 'meal_deal_price': 5.5,
 'total_individual_price': 8.55,
 'total_savings': 3.0500000000000003,
 'total_calories': 691.0,
 'total_protein': 51.4,
 'total_sugar': 24.3,
 'total_fat': 22.6,
 'total_carbs': 69.4,
 'total_salt': 3.9000000000000004,
 'objective_used': 'max_protein',
 'objective_value': 51.4}

In [59]:
params.update({
    "protein_preference": "others",

    "snack_preference": "no_preference",
    "main_type_preference": "no_preference",
    "cuisine_preference": "no_preference",
    "prime_preference": "no_preference"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 5 rows, 171 columns and 363 nonzeros (Max)
Model fingerprint: 0xeb84ce66
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]

Found heuristic solution: objective 11.0000000
Presolve removed 2 rows and 161 columns
Presolve time: 0.00s
Presolved: 3 rows, 10 columns, 18 nonzeros
Found heuristic solution: objective 28.2000000
Variable types: 0 continuous, 10 integer (10 binary)
Found heuristic solution: objective 34.3000000

Root relaxation: objective 6.150000e+01, 2 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Exp

{'main': 'Tesco Free Range Egg Mayonnaise Sandwich',
 'snack': 'Lindahls Protein Pudding Chocolate Brownie 140g',
 'drink': 'For Goodness Shakes Protein Strawberry Shake 250ml',
 'main_price': 2.2,
 'snack_price': 1.4,
 'drink_price': 2.25,
 'main_is_prime': 0,
 'meal_deal_price': 3.85,
 'total_individual_price': 5.85,
 'total_savings': 2.0,
 'total_calories': 678.0,
 'total_protein': 61.5,
 'total_sugar': 20.9,
 'total_fat': 15.8,
 'total_carbs': 69.3,
 'total_salt': 1.67,
 'objective_used': 'max_protein',
 'objective_value': 61.5}

In [60]:
params.update({
    "protein_preference": "fish",
    "main_type_preference": "wrap",
    "cuisine_preference": "caribbean"
})

solve_meal_deal(mains, snacks, drinks, params)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 23.6.0 23G80)

CPU model: Apple M2
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 171 columns and 377 nonzeros (Max)
Model fingerprint: 0xaa11cb81
Model has 156 linear objective coefficients
Variable types: 0 continuous, 171 integer (171 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+02]
  Objective range  [2e-02, 4e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]

Presolve removed 1 rows and 56 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 0
No other solutions better than -1e+100

Model is infeasible
Best objective -, best bound -, gap -
No optimal solution found.
Model status: 3
